In [ ]:
# Import TensorFlow for building and training the CNN model
import tensorflow as tf
# Import Keras layers and models for creating the CNN architecture
from tensorflow.keras import layers, models
# Import NumPy for numerical operations and finding predicted classes
import numpy as np
# Import load_dataset to download/load the tomato leaf disease dataset
from datasets import load_dataset
# Import os for creating directories and handling file paths
import os
# Import Image from PIL for saving the dataset images as JPG files
from PIL import Image

# Load the Tomato Leaf Disease dataset from Hugging Face
dataset = load_dataset("Project-AgML/tomato_leaf_disease")
print(dataset)

data = dataset["train"]
print("Total images:", len(data))

class_names = data.features["label"].names
print("Classes:", class_names)
print("Number of classes:", len(class_names))

# Split the dataset into training and validation sets
split_dataset = data.train_test_split(
    test_size=0.2,
    seed=42
)

train_data = split_dataset["train"]
val_data = split_dataset["test"]

print("Training images:", len(train_data))
print("Validation images:", len(val_data))

base_dir = "tomato_dataset"

train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")


for class_name in class_names:

    # Create the folder for the current class in the training directory
    # exist_ok=True prevents an error if the folder already exists
    os.makedirs(
        os.path.join(train_dir, class_name),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(val_dir, class_name),
        exist_ok=True
    )

print("Saving training images...")

for i, item in enumerate(train_data):

    image = item["image"]
    label = item["label"]

    class_name = class_names[label]

    image_path = os.path.join(
        train_dir,
        class_name,
        f"image_{i}.jpg"
    )

    image.save(image_path)

print("Saving validation images...")

for i, item in enumerate(val_data):

    image = item["image"]
    label = item["label"]

    class_name = class_names[label]

    # Create the complete path where the image will be saved
    image_path = os.path.join(
        val_dir,
        class_name,
        f"image_{i}.jpg"
    )

    image.save(image_path)


print("Dataset preparation completed!")

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(128, 128),
    batch_size=32,
    shuffle=True,
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(128, 128),
    batch_size=32,
    shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

# Normalize the pixel values of the training images
#
# Original pixel values are generally in the range 0 to 255.
# Dividing by 255 converts them to the range 0 to 1.
train_ds = train_ds.map(
    lambda x, y: (
        tf.cast(x, tf.float32) / 255.0,
        y
    )
)

# Normalize the validation images in the same way as
# the training images
#
# Pixel values are converted from 0-255 to 0-1.
val_ds = val_ds.map(
    lambda x, y: (
        tf.cast(x, tf.float32) / 255.0,
        y
    )
)

model = models.Sequential([

    layers.Conv2D(
        32,
        (3, 3),
        activation="relu",
        input_shape=(128, 128, 3)
    ),

    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(
        64,
        (3, 3),
        activation="relu"
    ),

    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(
        128,
        (3, 3),
        activation="relu"
    ),

    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dense(
        len(class_names),
        activation="softmax"
    )
])

model.summary()
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

loss, accuracy = model.evaluate(val_ds)
print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)

for images, labels in val_ds.take(1):
    predictions = model.predict(images)
    predicted_class = np.argmax(predictions[0])
    actual_class = labels[0].numpy()
    print(
        "Predicted:",
        class_names[predicted_class]
    )
    print(
        "Actual:",
        class_names[actual_class]
    )
    break

c:\Users\Prachi Jadhav\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Prachi Jadhav\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Prachi Jadhav\.cache\huggingface\hub\datasets--Project-AgML--tomato_leaf_disease. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Dev

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 11000
    })
})
Total images: 11000
Classes: ['Bacterial Spot', 'Early Blight', 'Healthy', 'Late Blight', 'Leaf Mold', 'Septoria Leaf Spot', 'Spider Mites Two-spotted Spider Mite', 'Target Spot', 'Tomato Mosaic Virus', 'Tomato Yellow Leaf Curl Virus']
Number of classes: 10
Training images: 8800
Validation images: 2200
Saving training images...
Saving validation images...
Dataset preparation completed!
Found 8800 files belonging to 10 classes.
Found 2200 files belonging to 10 classes.
Classes: ['Bacterial Spot', 'Early Blight', 'Healthy', 'Late Blight', 'Leaf Mold', 'Septoria Leaf Spot', 'Spider Mites Two-spotted Spider Mite', 'Target Spot', 'Tomato Mosaic Virus', 'Tomato Yellow Leaf Curl Virus']


c:\Users\Prachi Jadhav\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

 Trainable params: 3,305,930 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


c:\Users\Prachi Jadhav\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


275/275 ━━━━━━━━━━━━━━━━━━━━ 135s 469ms/step - accuracy: 0.5800 - loss: 1.2059 - val_accuracy: 0.7191 - val_loss: 0.8318
Epoch 2/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 2053s 7s/step - accuracy: 0.8161 - loss: 0.5399 - val_accuracy: 0.8359 - val_loss: 0.4650
Epoch 3/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 135s 488ms/step - accuracy: 0.8800 - loss: 0.3436 - val_accuracy: 0.8523 - val_loss: 0.4620
Epoch 4/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 1147s 4s/step - accuracy: 0.8995 - loss: 0.2828 - val_accuracy: 0.8559 - val_loss: 0.4422
Epoch 5/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 137s 497ms/step - accuracy: 0.9333 - loss: 0.1879 - val_accuracy: 0.8282 - val_loss: 0.5421
Epoch 6/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 139s 504ms/step - accuracy: 0.9418 - loss: 0.1661 - val_accuracy: 0.8623 - val_loss: 0.5427
Epoch 7/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 140s 507ms/step - accuracy: 0.9574 - loss: 0.1320 - val_accuracy: 0.8945 - val_loss: 0.3903
Epoch 8/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 145s 525ms/step - accuracy: 0.9651 - loss: 0.1011 -